In [8]:
import pandas as pd
import nltk
import re
import matplotlib.pyplot as plt
import numpy as np
from textblob import TextBlob
from sklearn.model_selection import train_test_split
import spacy
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import KFold
import torch
from torch.utils.data import DataLoader, TensorDataset

In [4]:
nlp = spacy.load("en_core_web_sm")

news_df = pd.read_csv('/content/drive/MyDrive/HLT project/Project files/sentiment_annotated_with_texts.csv')
df = news_df
# news_df.head()

/usr/local/lib/python3.10/dist-packages/spacy/util.py:1740: UserWarning: [W111] Jupyter notebook detected: if using `prefer_gpu()` or `require_gpu()`, include it in the same cell right before `spacy.load()` to ensure that the model is loaded on the correct device. More information: http://spacy.io/usage/v3#jupyter-notebook-gpu
  warnings.warn(Warnings.W111)


This section of code is used for preprocessing and data cleaning, collected from another notebook of ours.

In [5]:
# Handling missing values
news_df.fillna('', inplace=True)

# Removing duplicate rows
news_df = news_df.drop_duplicates()

# Converting to lowercase
news_df['title'] = news_df['title'].str.lower()
news_df['text'] = news_df['text'].str.lower()

# Removing punctuation
news_df['title'] = news_df['title'].apply(lambda x: re.sub(r'[^\w\s]', '', x))
news_df['text'] = news_df['text'].apply(lambda x: re.sub(r'[^\w\s]', '', x))

# Removing titles with less than 3 word, texts with less than 50 words
news_df = news_df[news_df['title'].apply(lambda x: len(x.split()) >= 3)]
news_df = news_df[news_df['text'].apply(lambda x: len(x.split()) >= 50)]

# Let's use FinBert!

There are many models of FinBert from HuggingFace, and we will use "ProsusAI".

In [9]:
# Use a pipeline as a high-level helper
from transformers import pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
torch.device(device)

pipe = pipeline("text-classification", model="ProsusAI/finbert", truncation=True)

cuda


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

As we got our model, we will run it without any other feature added, just as pretrained as it is.

In [10]:
x = []
with torch.no_grad():
  for i in news_df.index:
    x.append(pipe(news_df['title'][i]))

take only the label from the prediction, we are

In [11]:
filtered_x = [el[0]['label'] for el in x]

In [12]:
predictions = filtered_x
targets = list(news_df['true_sentiment'].apply(lambda a : a.lower()))
print(classification_report(targets, predictions, labels=["positive", "negative", "neutral"]))

              precision    recall  f1-score   support

    positive       0.60      0.61      0.61       765
    negative       0.56      0.68      0.61       709
     neutral       0.53      0.41      0.47       800

    accuracy                           0.57      2274
   macro avg       0.56      0.57      0.56      2274
weighted avg       0.56      0.57      0.56      2274



Let's try to fine-tune the model. The split train/test split will be 80/20.

In [19]:
from transformers import AdamW, BertTokenizer, AutoModelForSequenceClassification

convert_dict = {
    "positive": 0,
    "negative": 1,
    "neutral": 2
}
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.device(device)
model_name = "ProsusAI/finbert"

tokenizer = BertTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

texts = list(news_df["title"])
labels = [convert_dict[el.lower()] for el in list(news_df["true_sentiment"])]

tokenized_texts = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

# Convert labels to tensor
labels = torch.tensor(labels)
split_text = list(zip(tokenized_texts["input_ids"], tokenized_texts['token_type_ids'], tokenized_texts['attention_mask']))

train_inputs, validation_inputs, train_labels, validation_labels = train_test_split([i for i in range(len(labels))], torch.Tensor(labels), test_size=0.2, random_state=42)

train_inputs = {
    "input_ids": tokenized_texts['input_ids'][train_inputs],
    "attention_mask": tokenized_texts['attention_mask'][train_inputs],
    "token_type_ids": tokenized_texts['token_type_ids'][train_inputs]
}

validation_inputs = {
    "input_ids": tokenized_texts['input_ids'][validation_inputs],
    "attention_mask": tokenized_texts['attention_mask'][validation_inputs],
    "token_type_ids": tokenized_texts['token_type_ids'][validation_inputs]
}

# train and test dataset and dataloader
train_dataset = TensorDataset(train_inputs['input_ids'], train_inputs['attention_mask'], train_labels)
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)

validation_dataset = TensorDataset(validation_inputs['input_ids'], validation_inputs['attention_mask'], validation_labels)
validation_dataloader = DataLoader(validation_dataset, batch_size=16, shuffle=False)

# we will use ADAM for our problem
optimizer = AdamW(model.parameters(), lr=5e-5)
min_loss = 1000.0
# Training loop
epochs = 10
for epoch in range(epochs):
    model.train()
    for batch in train_dataloader:
        optimizer.zero_grad()
        input_ids, attention_mask, labels = batch
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in validation_dataloader:
            input_ids, attention_mask, labels = batch
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            val_loss += loss.item()
            _, predicted = torch.max(outputs.logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    if val_loss < min_loss:
      # Save the best fine-tuned model
      model.save_pretrained("/content/drive/MyDrive/HLT project/Models/finbert_model")
      min_loss = val_loss
    print(f"Epoch {epoch+1}/{epochs}, Validation Loss: {val_loss}, Accuracy: {accuracy}")

Epoch 1/10, Validation Loss: 19.040786743164062, Accuracy: 0.7318681318681318
Epoch 2/10, Validation Loss: 19.011122822761536, Accuracy: 0.734065934065934
Epoch 3/10, Validation Loss: 19.90372183918953, Accuracy: 0.7318681318681318
Epoch 4/10, Validation Loss: 22.315105959773064, Accuracy: 0.7582417582417582
Epoch 5/10, Validation Loss: 27.396062210202217, Accuracy: 0.7714285714285715
Epoch 6/10, Validation Loss: 29.804155737161636, Accuracy: 0.7406593406593407
Epoch 7/10, Validation Loss: 31.536600137129426, Accuracy: 0.7362637362637363
Epoch 8/10, Validation Loss: 33.921720994636416, Accuracy: 0.7538461538461538
Epoch 9/10, Validation Loss: 35.61907038092613, Accuracy: 0.7406593406593407
Epoch 10/10, Validation Loss: 41.523755541071296, Accuracy: 0.7296703296703296


A classification report for the test data:

In [29]:
# using the best model
model = AutoModelForSequenceClassification.from_pretrained(r"/content/drive/MyDrive/HLT project/Models/finbert_model")

model.eval()
predictions = []
true_labels = []

for batch in validation_dataloader:
  with torch.no_grad():
    input_ids, attention_mask, labels = batch
    outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
    _, predicted = torch.max(outputs.logits, 1)
    predictions.extend(predicted.tolist())
    true_labels.extend(labels.tolist())

print("Classification Report:")
print(classification_report(true_labels, predictions))

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.66      0.74       155
           1       0.78      0.75      0.76       139
           2       0.64      0.79      0.71       161

    accuracy                           0.73       455
   macro avg       0.75      0.73      0.74       455
weighted avg       0.75      0.73      0.74       455

